# 02 — Pré-processamento

Este notebook trata nulos, duplicidades e o código sentinela, define a variável alvo, cria as
features, confere se há vazamento e salva a base de modelagem em `data/processed/`. A padronização
e a codificação ficam dentro do `Pipeline` do notebook 03, para serem ajustadas só com os dados de
treino.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

sys.path.insert(0, str(Path("..").resolve()))
from src import config
assert config.RANDOM_STATE == RANDOM_STATE

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")
config.FIGURES.mkdir(parents=True, exist_ok=True)

In [2]:
from src.data import load_application, load_credit, save_processed
from src.preprocessing import (build_features, build_history_features, build_target,
                               check_leakage, check_missing, remove_duplicate_ids)

application = load_application()
credit = load_credit()

## 1. Dados faltantes

In [3]:
display(check_missing(application))
display(check_missing(credit))

,nulos,pct
OCCUPATION_TYPE,134203,30.6000


,nulos,pct


In [4]:
nulo = application["OCCUPATION_TYPE"].isna()
print("Distribuição do tipo de renda entre os que NÃO informaram ocupação:")
print(application.loc[nulo, "NAME_INCOME_TYPE"].value_counts(normalize=True).round(3).to_string())

Distribuição do tipo de renda entre os que NÃO informaram ocupação:
NAME_INCOME_TYPE
Pensioner              0.5620
Working                0.2670
Commercial associate   0.1250
State servant          0.0460
Student                0.0000


**Decisão:** o único campo com nulos é `OCCUPATION_TYPE`, com 134.203 linhas (30,6%). Os nulos não
são aleatórios: **56,2% deles são aposentados**, que naturalmente não informam ocupação.

- **Imputar pela moda** ("Laborers") seria errado, porque transformaria aposentados em operários.
- **Remover as linhas** descartaria quase um terço da base.

Por isso o nulo vira a categoria **`Nao_informado`**, que preserva a informação de que o campo não
foi preenchido. O histórico de crédito não tem nenhum nulo.

## 2. Duplicidades

In [5]:
print("Linhas 100% idênticas:", application.duplicated().sum())
repetidos = application[application["ID"].duplicated(keep=False)].sort_values("ID")
print("Linhas com ID repetido:", len(repetidos), "| IDs distintos envolvidos:", repetidos["ID"].nunique())
colunas_que_mudam = (repetidos.groupby("ID").nunique() > 1).sum()
print("Colunas que diferem entre cópias do mesmo ID:")
print(colunas_que_mudam[colunas_que_mudam > 0].to_string())

application = remove_duplicate_ids(application)
print("\nCadastro após remover IDs repetidos:", application.shape)

Linhas 100% idênticas: 0
Linhas com ID repetido: 94 | IDs distintos envolvidos: 47
Colunas que diferem entre cópias do mesmo ID:
CODE_GENDER            26
FLAG_OWN_CAR           21
FLAG_OWN_REALTY        23
CNT_CHILDREN           19
AMT_INCOME_TOTAL       47
NAME_INCOME_TYPE       30
NAME_EDUCATION_TYPE    17
NAME_FAMILY_STATUS     30
NAME_HOUSING_TYPE       9
DAYS_BIRTH             47
DAYS_EMPLOYED          47
FLAG_WORK_PHONE        17
FLAG_PHONE             21
FLAG_EMAIL             11
OCCUPATION_TYPE        20
CNT_FAM_MEMBERS        28

Cadastro após remover IDs repetidos: (438510, 18)


**Decisão:** não existem linhas 100% idênticas, mas há **47 IDs repetidos (94 linhas)**. Nos 47
casos, renda, data de nascimento e tempo de emprego mudam entre as cópias, e em 26 até o gênero é
diferente. Na prática são pessoas diferentes com o mesmo ID, o que é um problema de qualidade da
fonte.

Como não há critério para decidir qual cópia é a correta, mantemos **a primeira ocorrência**. O
impacto é desprezível: são 47 de 438.510 IDs.

## 3. Definição da variável alvo

A base não traz uma coluna "aprovado/reprovado". O alvo é construído a partir do histórico de
pagamento de cada cliente, e o limiar precisa ser justificado pela distribuição. Por isso começamos
olhando o **pior status** que cada cliente já teve.

In [6]:
ordem = {"X": -2, "C": -1, "0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5}
pior = credit.assign(nivel=credit["STATUS"].map(ordem)).groupby("ID")["nivel"].max()
rotulos = {-2: "X (sem empréstimo)", -1: "C (quitado)", 0: "0 (1-29 dias)", 1: "1 (30-59 dias)",
           2: "2 (60-89 dias)", 3: "3 (90-119 dias)", 4: "4 (120-149 dias)", 5: "5 (150+ / prejuízo)"}
dist = pior.value_counts().sort_index()
tabela = pd.DataFrame({"clientes": dist.values,
                       "pct": (dist.values / dist.sum() * 100).round(2),
                       "pct_acumulado_pior_ou_igual": (dist[::-1].cumsum()[::-1].values / dist.sum() * 100).round(2)},
                      index=[rotulos[i] for i in dist.index])
tabela.index.name = "pior status do cliente"
tabela

,clientes,pct,pct_acumulado_pior_ou_igual
pior status do cliente,,,
X (sem empréstimo),4536,9.8600,100.0000
C (quitado),1417,3.0800,90.1400
0 (1-29 dias),34682,75.4200,87.0500
1 (30-59 dias),4683,10.1800,11.6300
2 (60-89 dias),336,0.7300,1.4500
3 (90-119 dias),88,0.1900,0.7200
4 (120-149 dias),48,0.1000,0.5300
5 (150+ / prejuízo),195,0.4200,0.4200


**Definição adotada:** `mau_pagador = 1` quando o cliente teve **pelo menos um mês com atraso de 60
dias ou mais** (STATUS 2, 3, 4 ou 5) no histórico observado. Caso contrário, `mau_pagador = 0`.

**Justificativa pela distribuição do pior status de cada cliente:**

- **Atraso curto de 1 a 29 dias (status 0):** é o pior status de **75,4%** dos clientes. É
  praticamente o comportamento normal (esquecimento, data de vencimento) e não indica risco de
  crédito.
- **Corte em 30 dias (status 1 ou pior):** marcaria **11,6%** dos clientes como maus pagadores.
  Muitos deles atrasaram uma única fatura e regularizaram depois.
- **Corte em 60 dias (status 2 ou pior):** marca **1,45%** dos clientes. Isso significa pelo menos
  **duas faturas consecutivas sem pagamento**, um comportamento já associado a dificuldade
  financeira real e próximo da noção de inadimplência usada no mercado (que costuma considerar 90
  dias para default).

O limiar de 60 dias separa o atraso pontual do atraso que indica risco. Ele é uma **hipótese de
modelagem**, não uma política real de crédito. Um corte de 30 dias é uma alternativa legítima, que
aparece nas limitações do notebook 04.

In [7]:
alvo = build_target(credit)
print(alvo[config.TARGET].value_counts().to_string())
print(alvo[config.TARGET].value_counts(normalize=True).round(4).to_string())

mau_pagador
0    45318
1      667
mau_pagador
0   0.9855
1   0.0145


## 4. Checagem de vazamento

In [8]:
corr_vazamento = check_leakage(credit, alvo)
print(f"Correlação entre 'teve mês em atraso de 60+ dias' e o alvo: {corr_vazamento:.4f}")

Correlação entre 'teve mês em atraso de 60+ dias' e o alvo: 1.0000


**Vazamento de dados encontrado e evitado:** uma feature natural seria "quantos meses o cliente
ficou em atraso de 60 dias ou mais". Mas ela tem **correlação 1,0 com o alvo**, porque é a própria
regra do alvo escrita de outro jeito. Na primeira versão deste projeto ela chegou a ser usada, e os
quatro modelos alcançaram ROC-AUC = 1,0. Um resultado perfeito desses é sinal de bug, não de
sucesso. A feature foi removida.

As features de histórico mantidas (`meses_observados`, `meses_quitado`, `meses_sem_emprestimo`)
**não usam os status de atraso** (2 a 5).

## 5. Feature engineering

In [9]:
historico = build_history_features(credit)
historico.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,"45,985.0000","5,070,163.9131","45,433.6359","5,001,711.0000","5,026,147.0000","5,065,737.0000","5,114,024.0000","5,150,487.0000"
meses_observados,"45,985.0000",22.8025,15.4928,1.0000,10.0000,19.0000,34.0000,61.0000
meses_quitado,"45,985.0000",9.6125,13.6252,0.0000,0.0000,0.0000,16.0000,60.0000
meses_sem_emprestimo,"45,985.0000",4.5500,9.6227,0.0000,0.0000,1.0000,3.0000,61.0000


In [10]:
cadastro = build_features(application)
print("Colunas removidas: DAYS_BIRTH, DAYS_EMPLOYED (convertidas em anos), FLAG_MOBIL (constante)")
print("Colunas criadas: idade_anos, anos_empregado, sem_vinculo_emprego")
cadastro[["idade_anos", "anos_empregado", "sem_vinculo_emprego"]].describe().T

Colunas removidas: DAYS_BIRTH, DAYS_EMPLOYED (convertidas em anos), FLAG_MOBIL (constante)
Colunas criadas: idade_anos, anos_empregado, sem_vinculo_emprego


,count,mean,std,min,25%,50%,75%,max
idade_anos,"438,510.0000",43.8002,11.4578,20.5000,34.3000,42.8000,53.3000,69.0000
anos_empregado,"363,186.0000",7.1817,6.5698,0.0000,2.5000,5.3000,9.6000,48.0000
sem_vinculo_emprego,"438,510.0000",0.1718,0.3772,0.0000,0.0000,0.0000,0.0000,1.0000


**Justificativa das features criadas:**

- **`idade_anos` e `anos_empregado`:** a base guarda idade e tempo de emprego como dias negativos a
  partir da data da solicitação. Em anos, as variáveis ficam legíveis na interpretação (SHAP e
  apresentação) sem mudar a informação.
- **`sem_vinculo_emprego`:** marca o código 365243 (aposentados). O tempo de emprego desses
  clientes fica vazio e é imputado pela mediana dentro do Pipeline.
- **`meses_observados`:** tempo de relacionamento com o banco, em meses de histórico.
- **`meses_quitado`:** meses em que o cliente quitou o saldo (status C).
- **`meses_sem_emprestimo`:** meses sem crédito em uso (status X).

**Seleção de atributos:**

- **`FLAG_MOBIL` sai** por ter variância zero (é igual para todos).
- **`ID` sai** por ser só um identificador.
- **`CNT_CHILDREN` e `CNT_FAM_MEMBERS` ficam, apesar da correlação de 0,83:** a redundância não
  prejudica os modelos de árvore, e na Regressão Logística a regularização padrão atenua a
  colinearidade.
- As demais variáveis são mantidas, e a importância de cada uma é medida depois do treino
  (notebook 04).

## 6. Junção das tabelas

In [11]:
base = (cadastro
        .merge(historico, on="ID", how="inner", validate="one_to_one")
        .merge(alvo, on="ID", how="inner", validate="one_to_one"))
print("Base de modelagem:", base.shape)
print("IDs únicos:", base["ID"].is_unique)
print("Cadastros sem histórico de crédito (ficam de fora):", len(cadastro) - len(base))
print(base[config.TARGET].value_counts(normalize=True).round(4).to_string())
display(check_missing(base))

Base de modelagem: (36457, 22)
IDs únicos: True
Cadastros sem histórico de crédito (ficam de fora): 402053
mau_pagador
0   0.9831
1   0.0169


,nulos,pct
anos_empregado,6135,16.8300


**Resultado:** a base de modelagem tem **36.457 clientes × 22 colunas**, um cliente por linha
(IDs únicos), com **1,69% de maus pagadores**.

**402.053 cadastros ficam de fora** porque não têm histórico de crédito. Sem histórico não há
como saber se foram bons ou maus pagadores, então eles não podem ser usados para treinar um modelo
supervisionado.

O único nulo restante é `anos_empregado`, em 6.135 linhas (16,8%): são os aposentados, tratados
pela flag `sem_vinculo_emprego` e pela imputação dentro do Pipeline.

## 7. Normalização / padronização

**Escolha:** `StandardScaler` nas variáveis numéricas e `OneHotEncoder` nas categóricas, precedidos
de `SimpleImputer` (mediana nas numéricas). Tudo isso fica em `src/models.py::build_preprocessor`
e é aplicado **dentro do `Pipeline`** do notebook 03.

**Por quê:** as escalas são muito diferentes (renda na casa das centenas de milhares, filhos entre
0 e 19). A Regressão Logística é sensível a isso, porque os coeficientes e a regularização dependem
da escala. Árvores, Random Forest e XGBoost não precisam de escala, mas passam pelo mesmo
pré-processamento para a comparação ser justa. Ajustar o scaler antes do split faria as
estatísticas do teste vazarem para o treino.

## 8. Salvar dataset tratado

In [12]:
save_processed(base, config.PROCESSED_FILE)
print("Salvo em", config.DATA_PROCESSED / f"{config.PROCESSED_FILE}.csv")
base.head()

Salvo em C:\projetoclaude\tech-challenge-cartao-credito\data\processed\base_modelagem.csv


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,idade_anos,sem_vinculo_emprego,anos_empregado,meses_observados,meses_quitado,meses_sem_emprestimo,mau_pagador
0,5008804,M,Y,Y,0,"427,500.0000",Working,Higher education,Civil marriage,Rented apartment,1,0,0,Nao_informado,2.0000,32.9000,0,12.4000,16,13,1,0
1,5008805,M,Y,Y,0,"427,500.0000",Working,Higher education,Civil marriage,Rented apartment,1,0,0,Nao_informado,2.0000,32.9000,0,12.4000,15,12,1,0
2,5008806,M,Y,Y,0,"112,500.0000",Working,Secondary / secondary special,Married,House / apartment,0,0,0,Security staff,2.0000,58.8000,0,3.1000,30,7,16,0
3,5008808,F,N,Y,0,"270,000.0000",Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0000,52.3000,0,8.4000,5,0,3,0
4,5008809,F,N,Y,0,"270,000.0000",Commercial associate,Secondary / secondary special,Single / not married,House / apartment,0,1,1,Sales staff,1.0000,52.3000,0,8.4000,5,0,5,0
